# Hematopoiesis case study: Boolean network analyses

This notebook analyses the subset-minimal Boolean networks inferred by scBOLT for the Nestorowa hematopoiesis case study. It compares their selected components with literature models, evaluates functional enrichment, visualises their shared regulatory structure, and inspects their attractor landscapes.

### Table of contents

1. [Data overview](#data)
2. [Reference-model comparison](#references)
3. [Gene selection enrichment analysis](#enrichment)
4. [Hematopoietic gene recovery](#recovery)
5. [Structural properties](#structure)
6. [Reachable attractors](#attractors)

### Figure index <a class="anchor" id="figure-index"></a>

| Notebook section | Generated file |
| --- | --- |
| [Structural properties: function families](#fig-function-families) | `ig_function_families.pdf` |
| [Structural properties: feedback core](#fig-feedback-core) | `ig_feedback_core.pdf` |


### Notebook setup


In [ ]:
%matplotlib inline

import gzip
import logging
import math
import os
import shutil
import tempfile
import urllib.request
import warnings
from itertools import combinations_with_replacement
from pathlib import Path

import bonesistools as bt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.obo_parser import GODag
from IPython.display import SVG, display

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
bt.omics.pl.set_default_params()


def find_hematopoiesis_dir(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        candidates = (path, path / "hematopoiesis")
        for candidate in candidates:
            if (candidate / "scripts" / "build_notebook_data.py").is_file():
                return candidate
    raise FileNotFoundError(f"Could not locate hematopoiesis from {start}")


HEMATOPOIESIS_DIR = find_hematopoiesis_dir()
CASE_STUDIES_DIR = HEMATOPOIESIS_DIR.parent
RESULTS_DIR = HEMATOPOIESIS_DIR / "results"
RESOURCES_DIR = HEMATOPOIESIS_DIR / "resources"
FIGURE_DIR = HEMATOPOIESIS_DIR / "figures"

GENEINFO_VERSION = "bundled"
DOROTHEA_API = "modern"
DOROTHEA_LEVELS = ["A", "B", "C"]
DOROTHEA_COMPATIBILITY = True
OMNIPATH_VERSION = "2025-08-13"
HCOP_VERSION = "bundled"

GO_RELEASE = "2026-03-25"
GO_CACHE_DIR = Path(tempfile.gettempdir()) / "scbolt-case-studies" / "go"
GO_BASIC_URL = f"https://release.geneontology.org/{GO_RELEASE}/ontology/go-basic.obo"
GENE2GO_URL = "https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz"
GO_CATEGORY_TO_NAMESPACE = {"Process": "BP", "Function": "MF", "Component": "CC"}

FIGURE_DIR.mkdir(exist_ok=True)

mouse_identifiers = bt.resources.ncbi.identifiers(
    organism="mouse",
    version=GENEINFO_VERSION,
)


### Helper functions


In [ ]:
def download_resource(url, destination):
    if destination.is_file():
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {destination.name}...")
    for attempt in range(1, 4):
        with tempfile.NamedTemporaryFile(
            dir=destination.parent,
            prefix=f".{destination.name}.",
            delete=False,
        ) as temporary_file:
            temporary_path = Path(temporary_file.name)

        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "scBOLT-case-study/1.0"},
            )
            with (
                urllib.request.urlopen(request) as response,
                temporary_path.open("wb") as output,
            ):
                shutil.copyfileobj(response, output)
            if destination.suffix == ".gz":
                with gzip.open(temporary_path, "rb") as archive:
                    while archive.read(1024 * 1024):
                        pass
            temporary_path.replace(destination)
            return destination
        except Exception:
            temporary_path.unlink(missing_ok=True)
            if attempt == 3:
                raise
            print(f"Retrying {destination.name} ({attempt}/3)...")


def get_go_resources():
    local_directories = (
        HEMATOPOIESIS_DIR / "resources" / "go",
        CASE_STUDIES_DIR / "apl" / "resources" / "go",
    )
    for directory in local_directories:
        go_basic = directory / "go_basic.obo"
        gene2go = directory / "gene2go.gz"
        if go_basic.is_file() and gene2go.is_file():
            return go_basic, gene2go

    return (
        download_resource(GO_BASIC_URL, GO_CACHE_DIR / f"go_basic_{GO_RELEASE}.obo"),
        download_resource(GENE2GO_URL, GO_CACHE_DIR / "gene2go.gz"),
    )


def read_mouse_gene2go(path, tax_id=10090):
    opener = gzip.open if path.suffix == ".gz" else open
    rows = []
    with opener(path, "rt") as stream:
        next(stream)
        for line in stream:
            fields = line.rstrip("\n").split("\t")
            if len(fields) >= 8 and fields[0] == str(tax_id):
                rows.append((int(fields[1]), fields[2], fields[7]))
    return pd.DataFrame(rows, columns=["GeneID", "GO_ID", "Category"])


def to_gene_ids(genes, identifiers):
    converted = identifiers(
        sorted(set(genes)),
        input_type="name",
        output_type="gene_id",
    )

    return {
        int(gene_id)
        for gene_id in converted
        if isinstance(gene_id, (int, np.integer))
        or isinstance(gene_id, str)
        and gene_id.isnumeric()
    }


def to_gene_symbols(genes, identifiers):
    return set(
        identifiers(
            sorted(set(genes)),
            input_type="name",
            output_type="symbol",
        )
    )


## Data overview <a class="anchor" id="data"></a>

### Load Boolean networks and configurations

All numerical subset-minimal solutions exported by scBOLT are loaded as an ensemble. Their completed macrostate configurations are retained for the attractor analysis. The DoRothEA A/B/C prior is loaded with the same pinned resources and compatibility settings as `config/params-abc.yml`.


In [ ]:
bn_dirs = sorted(
    [
        path
        for path in (RESULTS_DIR / "bn").iterdir()
        if path.is_dir()
        and path.name.isdigit()
        and (path / "model.bnet").is_file()
        and (path / "configs.csv").is_file()
    ],
    key=lambda path: int(path.name),
)
if not bn_dirs:
    raise FileNotFoundError(
        f"No subset-minimal Boolean network found in {RESULTS_DIR / 'bn'}"
    )

bns = []
configs = []
for path in bn_dirs:
    bns.append(bt.logic.io.read_bnet(path / "model.bnet"))
    configs.append(
        bt.logic.io.read_hypercubes(path / "configs.csv", orientation="columns")
    )

bns = bt.logic.bn.BooleanNetworkEnsemble(*bns)
components = bns.components
selected_genes = (
    RESULTS_DIR / "gene_selection" / "selected_genes.txt"
).read_text(encoding="utf-8").splitlines()
if set(selected_genes) != set(components):
    raise ValueError(
        "The exported gene selection does not match the BN components."
    )

print(f"Boolean networks: {len(bns):,}")
print(f"components: {len(components):,}")

study_ids = to_gene_ids(selected_genes, mouse_identifiers)

grn = bt.resources.omnipath.dorothea(
    organism="mouse",
    levels=DOROTHEA_LEVELS,
    identifiers=mouse_identifiers,
    version=OMNIPATH_VERSION,
    hcop_version=HCOP_VERSION,
    compatibility=DOROTHEA_COMPATIBILITY,
    flavor=DOROTHEA_API,
)
background_ids = to_gene_ids(grn.nodes, mouse_identifiers)

missing_study_ids = study_ids - background_ids
if missing_study_ids:
    raise ValueError(
        f"{len(missing_study_ids)} scBOLT component Gene IDs are absent from "
        "the DoRothEA background."
    )


## Reference-model comparison <a class="anchor" id="references"></a>

The scBOLT component set is compared with three literature-derived hematopoietic gene sets and the Boolean network inferred with the original Chevalier selection algorithm. All mouse identifiers are standardised explicitly with the pinned NCBI `GeneIdentifiers` resource before intersections are computed.


In [ ]:
genes = {
    "Collombet": to_gene_symbols(
        {
            "Runx1",
            "Cebpa",
            "Ikzf1",
            "Pax5",
            "Ets1",
            "Csf1",
            "Il7",
            "Il7r",
            "Csf1r",
            "Id2",
            "Foxo1",
            "Cebpb",
            "Cd19",
            "Flt3",
            "Mac1",
            "Gfi1",
            "Spi1",
            "Ebf1",
            "E2a",
            "Egr2",
            "Mef2c",
        },
        mouse_identifiers,
    ),
    "Hamey": to_gene_symbols(
        {
            "Bptf",
            "Cbfa2t3h",
            "Erg",
            "Ets1",
            "Ets2",
            "Etv6",
            "Fli1",
            "Gata1",
            "Gata2",
            "Gata3",
            "Gfi1b",
            "Hhex",
            "Hoxa5",
            "Hoxa9",
            "Hoxb4",
            "Ikzf1",
            "Ldb1",
            "Lmo2",
            "Lyl1",
            "Meis1",
            "Mitf",
            "Myb",
            "Nfe2",
            "Nkx2.3",
            "Notch",
            "Pbx1",
            "Prdm16",
            "Runx1",
            "Smarcc1",
            "Tal1",
            "Tcf7",
        },
        mouse_identifiers,
    ),
    "Moignard": to_gene_symbols(
        {
            "Etv2",
            "Fli1",
            "Scl",
            "Gata1",
            "Notch1",
            "Sox7",
            "Hoxb4",
            "Lyl1",
            "Ikaros",
            "Erg",
            "PU.1",
            "Myb",
            "Nfe2",
            "Ets1",
            "Eto2",
            "Hhex",
            "Lmo2",
            "Sox17",
            "Gfi1",
            "Gfi1b",
        },
        mouse_identifiers,
    ),
}

genes["scBOLT"] = set(components)

reference_bn = bt.logic.io.read_bnet(
    RESOURCES_DIR / "models" / "chevalier.bnet"
)
reference_bn = mouse_identifiers.convert_boolean_network(
    reference_bn,
    copy=True,
)
genes["Chevalier"] = set(reference_bn)

gene_intersection = pd.DataFrame(
    data=math.nan,
    index=["Collombet", "Hamey", "Moignard", "Chevalier", "scBOLT"],
    columns=["Collombet", "Hamey", "Moignard", "Chevalier", "scBOLT"],
    dtype=pd.Int64Dtype(),
)

for a, b in combinations_with_replacement(gene_intersection.index, 2):
    gene_intersection.at[a, b] = len(genes[a].intersection(genes[b]))

nodes = set(grn.nodes)

genes_in_database = pd.Series(
    data=math.nan,
    index=["Collombet", "Hamey", "Moignard", "Chevalier"],
    dtype=pd.Int64Dtype(),
)

for i in genes_in_database.index:
    genes_in_database[i] = len(nodes.intersection(genes[i]))

In [ ]:
display(gene_intersection)
display(genes_in_database)

## Gene selection enrichment analysis <a class="anchor" id="enrichment"></a>

The scBOLT components are tested for Gene Ontology enrichment using the mouse DoRothEA A/B/C network used for inference as the background. Gene symbols are converted to NCBI Gene IDs with the same pinned identifier resource.


In [ ]:
print("study size:", len(study_ids))
print("background size:", len(background_ids))
print("study in background:", len(study_ids & background_ids))
print("missing from background:", len(study_ids - background_ids))

In [ ]:
go_basic_file, gene2go_file = get_go_resources()
with Path(os.devnull).open("w") as log:
    go_dag = GODag(
        obo_file=str(go_basic_file),
        prt=log,
        optional_attrs={"relationship"},
    )

gene2go = read_mouse_gene2go(gene2go_file)
associations = {
    namespace: (
        gene2go.loc[gene2go["Category"] == category]
        .groupby("GeneID")["GO_ID"]
        .apply(set)
        .to_dict()
    )
    for category, namespace in GO_CATEGORY_TO_NAMESPACE.items()
}

for namespace, geneid2go in associations.items():
    print(f"{namespace} {len(geneid2go):,} annotated mouse genes")

In [ ]:
with Path(os.devnull).open("w") as log:
    goea = GOEnrichmentStudyNS(
        pop=background_ids,
        ns2assoc=associations,
        godag=go_dag,
        propagate_counts=True,
        alpha=0.05,
        methods=["fdr_bh"],
        log=log,
    )

In [ ]:
with Path(os.devnull).open("w") as log:
    goea_results = goea.run_study(
        study_ids=study_ids,
        log=log,
    )
goea_significant_results = [
    result
    for result in goea_results
    if result.p_fdr_bh < 0.05 and result.enrichment == "e"
]
goea_df = pd.DataFrame([vars(result) for result in goea_significant_results])

In [ ]:
goea_df.sort_values(by="p_fdr_bh", axis=0, ascending=True, inplace=True)

bp_results = goea_df[goea_df["NS"] == "BP"].copy()
bp_results = bp_results.set_index(["GO", "name"])
bp_results.index = pd.Index([" ".join(col) for col in bp_results.index])
bp_results = -np.log10(bp_results["p_fdr_bh"])

In [ ]:
display(bp_results.head(30))

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1)
plt.barh(
    bp_results.index,
    bp_results.values,
    facecolor=bt.omics.pl.get_color("pink"),
    edgecolor=bt.omics.pl.get_color("red"),
    linewidth=1.5,
)
ax.yaxis.set_label_position("right")
ax.yaxis.tick_right()
ax.set_xlabel(r"$-\log_{10}(p)$")
plt.gca().invert_yaxis()
plt.tight_layout()
fig.set_size_inches(12, 5)
fig.subplots_adjust(right=0.55)
ax.yaxis.set_ticks_position("none")
plt.show()

## Hematopoietic gene recovery <a class="anchor" id="recovery"></a>

GO-derived hematopoietic marker sets are used to compare absolute marker recovery across the literature models, the original Chevalier solution, and scBOLT.


In [ ]:
def get_genes_from_go(go, go2gene, go_dag):
    genes = set()

    go_terms = set(go) if isinstance(go, (list, set, tuple)) else {go}

    expanded_terms = set()
    for term in go_terms:
        if term in go_dag:
            expanded_terms.add(term)
            expanded_terms.update(go_dag[term].get_all_children())

    for term in expanded_terms:
        genes.update(go2gene.get(term, set()))

    return genes


gene2go = gene2go.copy()
gene2go["gene"] = mouse_identifiers(
    list(gene2go["GeneID"].astype(str)),
    input_type="gene_id",
    output_type="symbol",
)
# gene2go = gene2go.loc[gene2go["gene"].isin(background)]

go2gene = gene2go.groupby("GO_ID")["gene"].apply(lambda x: list(set(x))).to_dict()

In [ ]:
hematopoiesis_markers = set(get_genes_from_go(["GO:0030097"], go2gene, go_dag))
enlarged_hematopoiesis_markers = set(
    get_genes_from_go(
        [
            "GO:1903706",
            "GO:0030097",
            "GO:0060218",
            "GO:1902036",
            "GO:1902033",
            "GO:0035162",
            "GO:0002244",
            "GO:1901532",
            "GO:0002320",
            "GO:0030098",
            "GO:0046650",
            "GO:0046649",
            "GO:0002521",
            "GO:0002318",
            "GO:0030099",
            "GO:0061515",
            "GO:0045637",
            "GO:0030219",
            "GO:0035855",
            "GO:0045652",
            "GO:0030218",
            "GO:0048821",
            "GO:0045646",
            "GO:0036230",
            "GO:0071621",
            "GO:0030851",
            "GO:0030224",
            "GO:0042117",
            "GO:0002548",
            "GO:0045655",
        ],
        go2gene,
        go_dag,
    )
)
print(f"hematopoiesis marker size: {len(hematopoiesis_markers)}")
print(f"enlarged hematopoiesis marker size: {len(enlarged_hematopoiesis_markers)}")

In [ ]:
hematopoiesis_number = {}
for k, v in genes.items():
    hematopoiesis_number[k] = len(hematopoiesis_markers & v)

enlarged_hematopoiesis_number = {}
for k, v in genes.items():
    enlarged_hematopoiesis_number[k] = len(enlarged_hematopoiesis_markers & v)

In [ ]:
print(f"number of hematopoietic markers: {hematopoiesis_number}")
print(f"number of enlarged hematopoietic markers: {enlarged_hematopoiesis_number}")

In [ ]:
# Absolute hematopoietic gene recovery across models

x = [len(gene) for gene in genes.values()]
y = hematopoiesis_number.values()

colors = [
    bt.omics.pl.get_color("red") if model == "scBOLT" else bt.omics.pl.get_color("blue")
    for model in hematopoiesis_number
]

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(x, y, c=colors, alpha=1)

for k, xi, yi in zip(hematopoiesis_number, x, y):
    if k == "scBOLT":
        ax.text(xi, yi + 0.3, k, fontsize=14, ha="right", va="bottom")
    else:
        ax.text(
            xi + 3,
            yi,
            k,
            fontsize=14,
            ha="left",
            va="bottom" if k != "Chevalier" else "top",
        )

ax.set_xlim(0, max(x) + 5)
ax.set_ylim(0, max(y) + 2)
ax.set_yticks([5, 10, 15, 20])

ax.set_xlabel("Gene number", fontsize=16)
ax.set_ylabel("Hematopoietic gene number", fontsize=16)

plt.tight_layout()
plt.show()
# plt.savefig(
#    "hematopoietic_gene_recovery.pdf",
#    dpi=300,
#    bbox_inches="tight"
# )
plt.close()

In [ ]:
# Absolute hematopoietic gene recovery across models

x = [len(gene) for gene in genes.values()]
y = enlarged_hematopoiesis_number.values()

colors = [
    bt.omics.pl.get_color("red") if model == "scBOLT" else bt.omics.pl.get_color("blue")
    for model in enlarged_hematopoiesis_number
]

fig, ax = plt.subplots(figsize=(4, 4))

ax.scatter(x, y, c=colors, alpha=1)

for k, xi, yi in zip(enlarged_hematopoiesis_number, x, y):
    if k == "scBOLT":
        ax.text(xi, yi + 0.3, k, fontsize=14, ha="right", va="bottom")
    elif k == "Moignard":
        ax.text(xi + 3, yi + 3, k, fontsize=14, ha="left", va="bottom")
    else:
        ax.text(
            xi + 3,
            yi,
            k,
            fontsize=14,
            ha="left",
            va="bottom" if k != "Chevalier" else "top",
        )

ax.set_xlim(0, max(x) + 5)
ax.set_ylim(0, max(y) + 2)
# ax.set_yticks([5, 10, 15, 20])

ax.set_xlabel("Gene number", fontsize=16)
ax.set_ylabel("Hematopoietic gene number", fontsize=16)

plt.tight_layout()
plt.show()
# plt.savefig(
#    "hematopoietic_gene_recovery.pdf",
#    dpi=300,
#    bbox_inches="tight"
# )
plt.close()

In [ ]:
go_terms = {
    "HSC": ["GO:0060218", "GO:1902036", "GO:1902033", "GO:0035162"],
    "MPP": ["GO:0002244", "GO:1901532"],
    "LMPP": ["GO:0002320", "GO:0030098", "GO:0046650", "GO:0046649", "GO:0002521"],
    "CMP": ["GO:0002318", "GO:0030099", "GO:0061515", "GO:0045637"],
    "MEP": [
        "GO:0030219",
        "GO:0035855",
        "GO:0045652",
        "GO:0030218",
        "GO:0048821",
        "GO:0045646",
    ],
    "GMP": [
        "GO:0036230",
        "GO:0071621",
        "GO:0030851",
        "GO:0030224",
        "GO:0042117",
        "GO:0002548",
        "GO:0045655",
    ],
    "Hemopoiesis": ["GO:1903706", "GO:0030097"],
}

label_order = ["GMP", "MEP", "CMP", "LMPP", "MPP", "HSC", "Hemopoiesis"]

label_color = {
    "HSC": bt.omics.pl.get_color("black", color_type="hex"),
    "MPP": bt.omics.pl.get_color("gray", color_type="hex"),
    "LMPP": bt.omics.pl.get_color("teal", color_type="hex"),
    "CMP": bt.omics.pl.get_color("orange", color_type="hex"),
    "MEP": bt.omics.pl.get_color("darkred", color_type="hex"),
    "GMP": bt.omics.pl.get_color("gold", color_type="hex"),
    "Hemopoiesis": bt.omics.pl.get_color("blue", color_type="hex"),
}

marker_sets = {}
for go_name, go_set in go_terms.items():
    marker_sets[go_name] = set(get_genes_from_go(go_set, go2gene, go_dag))
    print(f"{go_name}: {len(marker_sets[go_name])}")

## Structural properties <a class="anchor" id="structure"></a>

The aggregated influence graph summarises the regulatory structures retained across the subset-minimal Boolean-network ensemble. Edge frequencies quantify how consistently each interaction is selected, while node summaries describe the stability or diversity of the inferred Boolean functions.

### Function-family overview <a class="anchor" id="fig-function-families"></a>

Structurally equivalent components are grouped into families to provide a compact overview of the complete inferred network ensemble.

In [ ]:
# Aggregated influence graph and function families
ig = bns.to_influence_graph()
shared_influences = sum(
    edge_data["count"] == len(bns) for _, _, edge_data in ig.edges(data=True)
)
print(f"aggregated influences: {ig.number_of_edges():,}")
print(f"shared by all networks: {shared_influences:,}")

function_family_options = {
    "collapse": "family",
    "drop_isolates": True,
    "edge_label": "frequency",
    "graph_attr": {
        "ratio": "compress",
        "overlap": "prism",
        "sep": "+0",
        "esep": "+0",
        "K": "0.35",
        "ranksep": "1.0",
        "pack": "true",
        "rankdir": "TB",
        "splines": "curve",
    },
    "node_attr": {"fontsize": "20"},
    "node_style": "stability",
    "edge_attr": {"fontsize": "20"},
    "edge_style": "frequency",
}

ig.show(**function_family_options, width="100%")
ig.to_pydot(**function_family_options).write_pdf(
    str(FIGURE_DIR / "ig_function_families.pdf")
)

### Feedback core <a class="anchor" id="fig-feedback-core"></a>

Feedback nodes are identified independently in each Boolean network, then combined to induce a subgraph of the ensemble-aggregated influence graph. Edge frequencies indicate the proportion of inferred Boolean networks containing each signed interaction; interactions connecting distinct feedback circuits are retained.

In [ ]:
feedback_nodes = bt.logic.ig.ensemble_feedback_nodes(
    *(network.to_influence_graph() for network in bns),
    include_selfloops=False,
)
feedback_ig = ig.copy()
feedback_ig.remove_nodes_from(set(feedback_ig) - feedback_nodes)
print(f"feedback nodes: {len(feedback_nodes):,}")

feedback_core_options = {
    "collapse": None,
    "edge_label": "frequency",
    "node_style": "count",
    "min_frequency": 0,
    "program": "dot",
    "graph_attr": {
        "ratio": "compress",
        "size": "10:10!",
        "rankdir": "TB",
        "splines": "curve",
        "ranksep": "0.5",
        "overlap": "prism",
        "nodesep": "0.40",
        "margin": "0",
        "pad": "0.12",
    },
    "node_attr": {"fontsize": "40"},
    "edge_attr": {"fontsize": "30"},
}

feedback_core = feedback_ig.to_pydot(**feedback_core_options)
for edge in feedback_core.get_edges():
    frequency = float(edge.get("frequency"))
    edge.set_penwidth("6")
    edge.set_label("" if frequency == 1 else f"{frequency:.3f}".rstrip("0").rstrip("."))

display(SVG(feedback_core.create_svg(prog="dot")))
feedback_core.write_pdf(
    str(FIGURE_DIR / "ig_feedback_core.pdf"),
    prog="dot",
)

## Reachable attractors <a class="anchor" id="attractors"></a>

For every subset-minimal Boolean network, the following cell counts all minimal trap spaces and the most-permissive attractors reachable from the completed `S1` and `S0` configurations. The summary reports the range across the ensemble, while the second table gives the complete count distribution.


In [ ]:
landscape_counts = []

for network_index, (network, config) in enumerate(zip(bns, configs), start=1):
    landscape_counts.append(
        {
            "network": network_index,
            "trap spaces": len(network.trap_spaces()),
            "attractors from S1": sum(
                1
                for _ in network.attractors(
                    config["S1"],
                    update="most-permissive",
                )
            ),
            "attractors from S0": sum(
                1
                for _ in network.attractors(
                    config["S0"],
                    update="most-permissive",
                )
            ),
        }
    )

landscape_counts = pd.DataFrame(landscape_counts).set_index("network")

landscape_summary = pd.DataFrame(
    {
        "minimum": landscape_counts.min(),
        "median": landscape_counts.median(),
        "maximum": landscape_counts.max(),
    }
)
display(landscape_summary)

landscape_distributions = pd.concat(
    {
        column: landscape_counts[column].value_counts().sort_index()
        for column in landscape_counts
    },
    axis="columns",
).fillna(0).astype(int)
landscape_distributions.index.name = "count"
display(landscape_distributions)
